In [ ]:
import geopandas
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
shp_df = geopandas.read_file('GeoFiles/CSB/CSB_AZ_cleaned.shp')
print(shp_df.columns)
shp_df["CSBID"] = shp_df["CSBID"].astype("int64")
print(shp_df.shape)
print(shp_df.crs)

In [ ]:
alfalfa_yield = pd.read_csv('Data/Alfalfa_Yield_Predictions_new.csv')
barley_yield = pd.read_csv('Data/Barley_Yield_Predictions_new.csv')
corn_yield = pd.read_csv('Data/Corn_Yield_Predictions_new.csv')
cotton_yield = pd.read_csv('Data/Cotton_Yield_Predictions_new.csv')
wheat_yield = pd.read_csv('Data/Wheat_Yield_Predictions_new.csv')

req = ['CSBID','Season_year','County','CDL', 'Shape_acers','Pred_Yield/acre','Pred_Yield_total','fb_bad','fb_good','target','Category']

alfalfa_yield = alfalfa_yield[req]
barley_yield = barley_yield[req]
corn_yield = corn_yield[req]
cotton_yield = cotton_yield[req]
wheat_yield = wheat_yield[req]

In [ ]:
# Drop double/ambiguous crop codes (avoid duplicate CSBID-year records)
DROP_CDL = {225, 237, 238}
alfalfa_yield = alfalfa_yield[~alfalfa_yield['CDL'].isin(DROP_CDL)]
barley_yield  = barley_yield[~barley_yield['CDL'].isin(DROP_CDL)]
corn_yield    = corn_yield[~corn_yield['CDL'].isin(DROP_CDL)]
cotton_yield  = cotton_yield[~cotton_yield['CDL'].isin(DROP_CDL)]
wheat_yield   = wheat_yield[~wheat_yield['CDL'].isin(DROP_CDL)]


In [ ]:
# Add new columns Crop to all
alfalfa_yield["Crop"] = "Alfalfa"
barley_yield["Crop"] = "Barley"
corn_yield["Crop"] = "Corn"
cotton_yield["Crop"] = "Cotton"
wheat_yield["Crop"] = "Wheat"

#Based on CSBID, adding geometry column to all
alfalfa_yield = alfalfa_yield.merge(shp_df[["CSBID","geometry"]], on="CSBID", how="left")
barley_yield = barley_yield.merge(shp_df[["CSBID","geometry"]], on="CSBID", how="left")
corn_yield = corn_yield.merge(shp_df[["CSBID","geometry"]], on="CSBID", how="left")
cotton_yield = cotton_yield.merge(shp_df[["CSBID","geometry"]], on="CSBID", how="left")
wheat_yield = wheat_yield.merge(shp_df[["CSBID","geometry"]], on="CSBID", how="left")

AZ_Yield = pd.concat([alfalfa_yield, barley_yield, corn_yield, cotton_yield, wheat_yield], ignore_index=True)

In [ ]:
# Rename columns with 10 character limit
AZ_Yield.rename(columns={
    "Season_year": "Year",
    "Shape_acers": "CSBACRES",
    "County": "CNTY",
    "Pred_Yield/acre": "PredYld_ar",
    "Pred_Yield_total": "PredYld"
}, inplace=True)

# ensuring correct data types and leading zeros for CSBID
AZ_Yield["CSBID"] = AZ_Yield["CSBID"].astype(str).str.zfill(15)

# Split into years and save as shapefiles
years = AZ_Yield["Year"].unique()
for year in years:
    out_path = f"GeoFiles/Farm_Yield/AZ_Yield_{year}.shp"
    gdf_year = geopandas.GeoDataFrame(AZ_Yield[AZ_Yield["Year"] == year], geometry="geometry", crs = shp_df.crs)
    gdf_year.to_file(out_path, driver="ESRI Shapefile")
    print(f"Saved {out_path} with shape {gdf_year.shape}")